# 08 · Regresión logística — modelo principal

**Objetivo.** Estimar los OR ajustados con el diseño muestral complejo, verificar el ajuste y exportar los coeficientes para el boletín y la API.

| | |
|---|---|
| **Entradas** | `data.parquet` |
| **Salidas** | `tabla_21` a `tabla_27`, `fig_11`, `fig_12`, `coeficientes_logistico.json` |

> Proyecto: *Modelamiento epidemiológico de la depresión en la población peruana de 15 y
> más años (ENDES 2025)* · Epidemiología 2026-2 · UNMSM


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "src"))
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "scripts"))

import numpy as np
import pandas as pd
from depresion import config as cfg, io as dio, viz
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
viz.aplicar_tema()
print("Proyecto:", cfg.RAIZ.name)

In [ ]:
%run ../scripts/analisis_05_logistica.py

## Crudo frente a ajustado

In [ ]:
pd.read_csv(cfg.TABLAS / "tabla_21_modelos_por_bloques.csv")[
    ["exposicion", "or_crudo", "or_ajustado", "cambio_pct", "cambia_de_direccion"]]

## Modelo completo

In [ ]:
log = pd.read_csv(cfg.TABLAS / "tabla_22_logistica_diseno_complejo.csv")
log[["etiqueta", "or_formato", "p", "significativo"]]

## Diagnóstico

In [ ]:
pd.read_csv(cfg.TABLAS / "tabla_24_diagnostico_logistica.csv")

## Conclusiones

- n = 28 553, 1 772 eventos, **EPV = 104**: holgadísimo frente al umbral clásico de 10.
- Se usa `svy.GLM`, que aplica los **grados de libertad del diseño** (2 994 = conglomerados
  menos estratos) y contrastes t y F, como `svyglm` de R.
- **Consumo problemático de alcohol: OR 6,07** (3,41-10,82), el más alto del modelo.
- Sexo femenino OR 2,94; edad 15-19 OR 2,11; Selva OR 0,67 frente a Costa.
- El tabaco pasa de 0,83 crudo a **1,44 ajustado**: cambia de dirección.
- Violencia física en la subpoblación con pareja: **OR 3,86** (2,74-5,44).
- AUC 0,667: discriminación aceptable. Sirve para priorizar poblaciones, no para decidir
  sobre una persona.
- Hosmer-Lemeshow rechaza (p = 0,006), pero con N grande es hipersensible; la curva de
  calibración y el Brier son la referencia.
- Box-Tidwell rechaza la linealidad del logit en la edad continua: **justifica usar el
  grupo etario categorizado**, que es lo que hace el modelo.
